**Comparaison des résultats & Synthèse ***

Ce notebook charge les métriques des 4 modèles et produit :
- Le tableau comparatif complet
- L'ablation curve (F1-macro progressif)
- Les courbes d'apprentissage des 3 modèles DL sur un même graphique
- Les 4 matrices de confusion côte à côte

**Imports & configuration**

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

import torch
from sklearn.metrics import confusion_matrix

RES_DIR  = Path('results')
DATA_DIR = Path('data/processed')

LABEL_NAMES = ['négatif', 'neutre', 'positif']

# Palette cohérente pour les 4 modèles
COLORS = {
    'baseline': '#7f8c8d',
    '01':      '#e67e22',
    '02':      '#2980b9',
    '03':      '#8e44ad'
}


**Chargement des métriques**

In [ ]:
# ── Charger les 4 fichiers JSON de métriques ──────────────────────────────────
with open(RES_DIR / 'metrics_baseline.json') as f:
    m_base = json.load(f)
with open(RES_DIR / 'metrics_01.json') as f:
    m_01 = json.load(f)
with open(RES_DIR / 'metrics_02.json') as f:
    m_02 = json.load(f)
with open(RES_DIR / 'metrics_03.json') as f:
    m_03 = json.load(f)

print('Métriques chargées :')
for name, m in [('Baseline', m_base), ('01', m_01), ('02', m_02), ('03', m_03)]:
    print(f'  {name:10s}  F1-macro = {m["f1_macro"]:.4f}  Accuracy = {m["accuracy"]:.4f}')

**Tableau cmparatif**

In [ ]:
# ── Construction automatique depuis les JSON ──────────────────────────────────
rows = []
ref_f1 = m_base['f1_macro']

for label, m in [
    ('Baseline TF-IDF',          m_base),
    ('01 FinBERT seul',          m_01),
    ('02 FinBERT + BiLSTM',      m_02),
    ('03 FinBERT + BiLSTM + Attn', m_03),
]:
    delta = m['f1_macro'] - ref_f1
    params = m.get('trainable_params', None)
    params_str = f"{params:,}" if params else '—'

    rows.append({
        'Modèle':        label,
        'Accuracy':      f"{m['accuracy']:.4f}",
        'F1-macro':      f"{m['f1_macro']:.4f}",
        'F1 négatif':    f"{m['f1_per_class']['negatif']:.4f}",
        'F1 neutre':     f"{m['f1_per_class']['neutre']:.4f}",
        'F1 positif':    f"{m['f1_per_class']['positif']:.4f}",
        'ROC-AUC':       f"{m.get('roc_auc', 0):.4f}",
        'Params entr.':  params_str,
        'Δ F1 vs base':  f"{delta:+.4f}" if delta != 0 else 'référence',
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

# Sauvegarder en CSV pour la soutenance
df_results.to_csv(RES_DIR / 'comparison_table.csv', index=False)
print('\nSauvegardé : results/comparison_table.csv')

**Ablation curve — F1-macro progressif**

In [ ]:
models_labels = [
    'Baseline\nTF-IDF',
    '01\nFinBERT',
    '02\n+BiLSTM',
    '03\n+Attention'
]
f1_values = [
    m_base['f1_macro'],
    m_01['f1_macro'],
    m_02['f1_macro'],
    m_03['f1_macro']
]
bar_colors = [COLORS['baseline'], COLORS['01'], COLORS['02'], COLORS['03']]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(models_labels, f1_values, color=bar_colors, width=0.55, edgecolor='white')

# Afficher la valeur au-dessus de chaque barre
for bar, val in zip(bars, f1_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f'{val:.4f}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

# Afficher les deltas entre barres consécutives
for i in range(1, len(f1_values)):
    delta = f1_values[i] - f1_values[i - 1]
    x_mid = (i - 0.5) + 0.5  # entre les barres i-1 et i
    y_pos = max(f1_values[i - 1], f1_values[i]) + 0.025
    color = '#27ae60' if delta > 0 else '#e74c3c'
    ax.annotate(
        f'{delta:+.4f}',
        xy=(i, f1_values[i]),
        xytext=(i - 0.5, y_pos),
        fontsize=9, color=color, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=color, lw=1.2)
    )

ax.set_ylim(min(f1_values) - 0.05, max(f1_values) + 0.07)
ax.set_ylabel('F1-macro (test)', fontsize=12)
ax.set_title('Ablation Study — Apport marginal de chaque composant', fontsize=13)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(RES_DIR / 'ablation_curve.png', dpi=150)
plt.show()
print('Sauvegardé : results/ablation_curve.png')

**Courbes d'apprentissage — 3 modèles DL sur un même graphique**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for key, m, label in [
    ('01', m_01, '01 FinBERT seul'),
    ('02', m_02, '02 + BiLSTM'),
    ('03', m_03, '03 + Attention'),
]:
    history = m.get('val_f1_history', [])
    if history:
        epochs = range(1, len(history) + 1)
        ax.plot(epochs, history, marker='o', color=COLORS[key], label=label, linewidth=2)

        # Marquer la meilleure époque
        best_ep = m.get('best_epoch', None)
        if best_ep and best_ep <= len(history):
            ax.scatter(
                best_ep, history[best_ep - 1],
                color=COLORS[key], s=120, zorder=5, marker='*'
            )

# Lignes de référence
ax.axhline(y=m_base['f1_macro'], color=COLORS['baseline'],
           linestyle='--', linewidth=1.5, label=f'Baseline ({m_base["f1_macro"]:.4f})')

ax.set_xlabel('Époque', fontsize=12)
ax.set_ylabel('Val F1-macro', fontsize=12)
ax.set_title('Courbes d\'apprentissage — Val F1-macro des 3 modèles DL', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(RES_DIR / 'learning_curves_comparison.png', dpi=150)
plt.show()
print('Sauvegardé : results/learning_curves_comparison.png')

**F1 par classe — comparaison des 4 modèles**

In [ ]:
# ── Barplot groupé : F1 par classe pour chaque modèle ────────────────────────
classes      = ['négatif', 'neutre', 'positif']
model_labels = ['Baseline', '01 FinBERT', '02 +BiLSTM', '03 +Attention']
model_keys   = ['baseline', '01', '02', '03']
metrics_list = [m_base, m_01, m_02, m_03]

x      = np.arange(len(classes))
width  = 0.18

fig, ax = plt.subplots(figsize=(10, 5))

for i, (key, m, label) in enumerate(zip(model_keys, metrics_list, model_labels)):
    f1s = [
        m['f1_per_class']['negatif'],
        m['f1_per_class']['neutre'],
        m['f1_per_class']['positif']
    ]
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, f1s, width, label=label, color=COLORS[key], alpha=0.85)

ax.set_xlabel('Classe', fontsize=12)
ax.set_ylabel('F1-score', fontsize=12)
ax.set_title('F1 par classe — Comparaison des 4 modèles', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(RES_DIR / 'f1_per_class_comparison.png', dpi=150)
plt.show()
print('Sauvegardé : results/f1_per_class_comparison.png')

**Matrices de confusion — 4 modèles côte à côte**

In [ ]:
# ── Recharger les prédictions test pour recalculer les matrices ───────────────
# On recharge les modèles et relance l'inférence sur le test set

from transformers import AutoModel
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Dataset minimal ───────────────────────────────────────────────────────────
class TweetDataset(Dataset):
    def __init__(self, encodings):
        self.input_ids      = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels         = encodings['labels']
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels':         self.labels[idx]
        }

test_enc    = torch.load(DATA_DIR / 'test_encodings.pt')
test_loader = DataLoader(TweetDataset(test_enc), batch_size=32, shuffle=False)
true_labels = test_enc['labels'].tolist()

print(f'Test set : {len(true_labels)} tweets chargés')

In [ ]:
# ── Définitions des 3 architectures (copie minimale depuis 01/02/03) ────────
BACKBONE = 'ProsusAI/finbert'

class FinBertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(BACKBONE)
        self.drop1      = nn.Dropout(0.3)
        self.dense      = nn.Linear(768, 256)
        self.relu       = nn.ReLU()
        self.drop2      = nn.Dropout(0.2)
        self.classifier = nn.Linear(256, 3)
    def forward(self, input_ids, attention_mask):
        cls = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.classifier(self.drop2(self.relu(self.dense(self.drop1(cls)))))


class FinBertBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(BACKBONE)
        self.bilstm     = nn.LSTM(768, 256, batch_first=True, bidirectional=True)
        self.drop1      = nn.Dropout(0.3)
        self.dense      = nn.Linear(512, 128)
        self.relu       = nn.ReLU()
        self.drop2      = nn.Dropout(0.2)
        self.classifier = nn.Linear(128, 3)
    def forward(self, input_ids, attention_mask):
        hs       = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        out, _   = self.bilstm(hs)
        last     = out[:, -1, :]
        return self.classifier(self.drop2(self.relu(self.dense(self.drop1(last)))))


class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W2 = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, lstm_output, attention_mask=None):
        scores = self.W2(torch.tanh(self.W1(lstm_output))).squeeze(-1)
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float('-inf'))
        alpha   = F.softmax(scores, dim=-1)
        context = torch.bmm(alpha.unsqueeze(1), lstm_output).squeeze(1)
        return context, alpha

class FinBertBiLSTMAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(BACKBONE)
        self.bilstm     = nn.LSTM(768, 256, batch_first=True, bidirectional=True)
        self.attention  = AttentionLayer(512)
        self.drop1      = nn.Dropout(0.3)
        self.dense      = nn.Linear(512, 128)
        self.relu       = nn.ReLU()
        self.drop2      = nn.Dropout(0.2)
        self.classifier = nn.Linear(128, 3)
    def forward(self, input_ids, attention_mask):
        hs          = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.bilstm(hs)
        context, _  = self.attention(lstm_out, attention_mask)
        return self.classifier(self.drop2(self.relu(self.dense(self.drop1(context)))))


print('Architectures définies.')

In [ ]:
def get_predictions(model, loader):
    """Inférence rapide — retourne les prédictions sur le test set."""
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                batch['input_ids'].to(DEVICE),
                batch['attention_mask'].to(DEVICE)
            )
            all_preds += logits.argmax(dim=-1).cpu().tolist()
    return all_preds


# ── Charger chaque modèle et récupérer ses prédictions ────────────────────────
all_preds = {}

configs = [
    ('01', FinBertClassifier,       'models/fintwit_classifier/best_model.pt'),
    ('02', FinBertBiLSTM,           'models/fintwit_bilstm/best_model.pt'),
    ('03', FinBertBiLSTMAttention,  'models/fintwit_bilstm_attention/best_model.pt'),
]

for key, ModelClass, ckpt_path in configs:
    print(f'Chargement {key}...', end=' ')
    m = ModelClass().to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    all_preds[key] = get_predictions(m, test_loader)
    del m
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    print('OK')

print('Prédictions récupérées pour 01 / 02 / 03')

In [ ]:
# ── Charger les prédictions de la baseline depuis son fichier de métriques ─────
# La baseline n'est pas un modèle PyTorch — on recharge depuis le CSV si disponible,
# sinon on reconstitue depuis sklearn
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

baseline_preds_path = Path('results/baseline_test_preds.npy')

if baseline_preds_path.exists():
    all_preds['baseline'] = np.load(baseline_preds_path).tolist()
    print('Prédictions baseline chargées depuis results/baseline_test_preds.npy')
else:
    # Recalculer depuis le modèle baseline sauvegardé
    try:
        with open('models/baseline_tfidf_logreg.pkl', 'rb') as f:
            baseline_model = pickle.load(f)
        df_test = pd.read_csv(DATA_DIR / 'test.csv')
        all_preds['baseline'] = baseline_model.predict(df_test['Tweet_clean'].fillna('')).tolist()
        print('Prédictions baseline recalculées depuis le modèle .pkl')
    except FileNotFoundError:
        print('⚠️  Modèle baseline non trouvé — matrice de confusion baseline indisponible.')
        print('    Seules les 3 matrices DL seront affichées.')
        all_preds['baseline'] = None

In [ ]:
# ── 4 matrices de confusion côte à côte ──────────────────────────────────────
plot_items = [
    ('baseline', 'Baseline TF-IDF',           'Greys'),
    ('01',      '01 FinBERT',                'Oranges'),
    ('02',      '02 +BiLSTM',               'Blues'),
    ('03',      '03 +Attention (final)',     'Purples'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

for ax, (key, title, cmap) in zip(axes, plot_items):
    preds = all_preds.get(key)
    if preds is None:
        ax.text(0.5, 0.5, 'Non disponible', ha='center', va='center', fontsize=12)
        ax.set_title(title)
        continue

    cm = confusion_matrix(true_labels, preds, normalize='true')
    sns.heatmap(
        cm, ax=ax, annot=True, fmt='.2%', cmap=cmap,
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
        linewidths=0.5, cbar=False
    )
    f1 = m_base['f1_macro'] if key == 'baseline' else locals().get(f'm_{key}', {}).get('f1_macro', 0)
    # Récupérer le bon f1 depuis les dicts
    f1_map = {'baseline': m_base, '01': m_01, '02': m_02, '03': m_03}
    f1 = f1_map[key]['f1_macro']

    ax.set_title(f'{title}\nF1-macro = {f1:.4f}', fontsize=11)
    ax.set_ylabel('Réel')
    ax.set_xlabel('Prédit')

plt.suptitle('Matrices de confusion — 4 modèles (test, normalisées)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RES_DIR / 'confusion_matrices_4models.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : results/confusion_matrices_4models.png')